# Prediksi Harga Mobil Bekas

Tugas Pengenalan Pola - Machine Learning (regresi)

**Tujuan:** memprediksi `price_usd` (harga mobil bekas) dari data mobil seperti tahun, mesin, merek, dll.

**Metrik:** MAE, RMSE, dan R2. Model harus lebih bagus dari baseline (prediksi rata-rata harga).

**Alur:**
1. Baca data dan cek data
2. Cleaning (missing value, duplikat, outlier)
3. EDA
4. Feature engineering
5. Split train dan test
6. Preprocessing (encoding dan scaling)
7. Bandingkan model pakai cross validation
8. Tuning
9. Evaluasi di data test
10. Simpan model dan deploy

## 1. Import library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

## 2. Baca data dan cek data

In [ ]:
df = pd.read_csv('used_car_prediction.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

Catatan: `mileage_kmpl` itu efisiensi bahan bakar (km per liter), bukan jarak tempuh.

## 3. Cleaning

In [ ]:
# cek missing value
df.isnull().sum()

In [ ]:
# cek duplikat
df.duplicated().sum()

Missing value cuma ada di `service_history`. Barisnya tidak dihapus karena jumlahnya cukup banyak, jadi diisi `Unknown` (artinya tidak ada catatan servis). Mengisi dengan nilai tetap seperti ini tidak bikin data leakage, jadi aman dilakukan sebelum split.

In [ ]:
df['service_history'] = df['service_history'].fillna('Unknown')
df.isnull().sum().sum()

In [ ]:
# cek outlier pakai boxplot
kolom_num = ['mileage_kmpl', 'engine_cc', 'owner_count', 'accidents_reported', 'price_usd']

plt.figure(figsize=(15, 3))
for i, col in enumerate(kolom_num):
    plt.subplot(1, 5, i + 1)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
# cek nilai yang tidak masuk akal
print('harga <= 0         :', (df['price_usd'] <= 0).sum())
print('tahun > 2026       :', (df['make_year'] > 2026).sum())
print('owner_count < 1    :', (df['owner_count'] < 1).sum())
print('accidents < 0      :', (df['accidents_reported'] < 0).sum())

In [ ]:
# cek fuel_type vs engine_cc (mobil listrik harusnya tidak punya cc)
df.groupby('fuel_type')[['engine_cc', 'mileage_kmpl']].describe().T

Kesimpulan cleaning: outlier tidak dihapus karena nilainya masih masuk akal (bukan salah input). Kalau di tabel terakhir mobil `Electric` ternyata punya `engine_cc`, itu masalah dari datanya, dicatat saja di keterbatasan.

## 4. EDA

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='price_usd', kde=True)
plt.title('Distribusi Harga Mobil')
plt.show()

print('skewness harga:', df['price_usd'].skew())

In [ ]:
df[['make_year', 'mileage_kmpl', 'engine_cc', 'owner_count', 'accidents_reported']].hist(figsize=(12, 7))
plt.show()

In [ ]:
# fitur numerik vs harga
plt.figure(figsize=(15, 8))
for i, col in enumerate(['make_year', 'engine_cc', 'mileage_kmpl', 'owner_count', 'accidents_reported']):
    plt.subplot(2, 3, i + 1)
    sns.scatterplot(data=df, x=col, y='price_usd', alpha=0.3)
    plt.title(col + ' vs harga')
plt.tight_layout()
plt.show()

In [ ]:
# fitur kategorikal vs harga
kolom_kat = ['brand', 'fuel_type', 'transmission', 'service_history', 'insurance_valid', 'color']

plt.figure(figsize=(16, 8))
for i, col in enumerate(kolom_kat):
    plt.subplot(2, 3, i + 1)
    sns.boxplot(data=df, x=col, y='price_usd')
    plt.xticks(rotation=45)
    plt.title(col + ' vs harga')
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('color')['price_usd'].agg(['count', 'mean', 'median']).sort_values('median', ascending=False)

In [ ]:
# apakah mobil baru dan mesin besar lebih mahal?
sns.scatterplot(data=df, x='make_year', y='price_usd', hue='engine_cc', palette='viridis')
plt.title('Tahun Produksi dan Kapasitas Mesin terhadap Harga')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

Catatan EDA (cocokkan dengan grafik di atas):
- Makin lama tahun pembuatan, harga makin turun, jadi nanti dibuat fitur umur mobil.
- Merek, jenis bahan bakar, transmisi, dan riwayat servis kelihatan beda harganya antar kategori.
- Warna selisihnya kecil, tapi tetap dipakai dulu. Nanti dilihat penting atau tidak di feature importance.
- Korelasi cuma mengukur hubungan linear dan hanya untuk kolom angka.

## 5. Feature engineering

Bikin kolom `car_age` (umur mobil) dari `make_year`, lalu `make_year` dibuang karena informasinya sama. Ini cuma hitungan biasa per baris (bukan belajar dari data), jadi aman dilakukan sebelum split.

In [ ]:
df['car_age'] = 2026 - df['make_year']
df = df.drop(columns=['make_year'])
df.head()

## 6. Split data

Data dibagi dulu sebelum encoding dan scaling supaya data test tidak ikut "dipelajari" (menghindari data leakage). Data test baru dipakai di akhir.

In [ ]:
X = df.drop(columns=['price_usd'])
y = df['price_usd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

## 7. Preprocessing

- kolom angka di-scale pakai StandardScaler
- kolom kategori di-encode pakai OneHotEncoder

Semua dimasukkan ke `Pipeline` bersama modelnya. Jadi saat cross validation, encoder dan scaler otomatis hanya fit di data train tiap fold (tidak bocor), dan nanti waktu deploy data mentah langsung bisa diprediksi.

In [ ]:
kolom_angka = X_train.select_dtypes(include='number').columns.tolist()
kolom_kat = X_train.select_dtypes(exclude='number').columns.tolist()

print(kolom_angka)
print(kolom_kat)

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), kolom_angka),
    ('cat', OneHotEncoder(handle_unknown='ignore'), kolom_kat)
])

# fungsi kecil supaya tidak nulis pipeline berulang
def buat_pipeline(model):
    return Pipeline([('prep', preprocessor), ('model', model)])

## 8. Bandingkan model (cross validation)

Kalau cuma satu kali split, hasilnya bisa kebetulan. Jadi dipakai cross validation 5 fold di data train. `Dummy` adalah baseline (selalu menebak rata-rata harga).

In [ ]:
models = {
    'Dummy (baseline)': buat_pipeline(DummyRegressor()),
    'Linear Regression': buat_pipeline(LinearRegression()),
    'Decision Tree': buat_pipeline(DecisionTreeRegressor(random_state=42)),
    'Random Forest': buat_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    'Gradient Boosting': buat_pipeline(GradientBoostingRegressor(random_state=42))
}

In [ ]:
results = []

for name, model in models.items():
    cv = cross_validate(model, X_train, y_train, cv=5, return_train_score=True,
                        scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'])
    results.append({
        'Model': name,
        'R2 Train': cv['train_r2'].mean(),
        'R2 CV': cv['test_r2'].mean(),
        'MAE CV': -cv['test_neg_mean_absolute_error'].mean(),
        'RMSE CV': -cv['test_neg_root_mean_squared_error'].mean()
    })

df_cv = pd.DataFrame(results).sort_values('RMSE CV').reset_index(drop=True)
df_cv.round(4)

Cara baca: bandingkan semua model dengan Dummy. Lalu lihat selisih R2 Train dan R2 CV, kalau jauh berarti overfitting. Decision Tree dan Random Forest yang belum di-tuning biasanya overfitting (R2 train hampir 1 tapi R2 CV jauh lebih kecil).

## 9. Hyperparameter tuning

Pakai GridSearchCV (cross validation 5 fold di data train) untuk model yang punya hyperparameter penting. Tujuannya mengurangi overfitting. Linear Regression tidak di-tuning karena tidak punya hyperparameter penting.

In [ ]:
grids = {
    'Decision Tree': (buat_pipeline(DecisionTreeRegressor(random_state=42)), {
        'model__max_depth': [3, 5, 8, 12],
        'model__min_samples_leaf': [1, 10, 30]
    }),
    'Random Forest': (buat_pipeline(RandomForestRegressor(random_state=42)), {
        'model__n_estimators': [100, 200],
        'model__max_depth': [5, 10, None],
        'model__min_samples_leaf': [1, 5, 10]
    }),
    'Gradient Boosting': (buat_pipeline(GradientBoostingRegressor(random_state=42)), {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [2, 3, 4]
    })
}

In [ ]:
tuned = {}

for name, (pipe, param_grid) in grids.items():
    print('tuning', name, '...')
    gs = GridSearchCV(pipe, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
    gs.fit(X_train, y_train)
    tuned[name] = gs
    print('  parameter terbaik:', gs.best_params_)
    print('  RMSE CV:', round(-gs.best_score_, 2))

In [ ]:
# bandingkan RMSE CV sebelum dan sesudah tuning
perbandingan = []
for name in tuned:
    sebelum = df_cv.loc[df_cv['Model'] == name, 'RMSE CV'].values[0]
    sesudah = -tuned[name].best_score_
    perbandingan.append({'Model': name, 'RMSE sebelum tuning': sebelum, 'RMSE sesudah tuning': sesudah})

pd.DataFrame(perbandingan).round(2)

## 10. Pilih model terbaik

Model dipilih dari hasil cross validation di data train (bukan dari data test). Yang RMSE CV-nya paling kecil yang dipilih.

In [ ]:
# kumpulkan RMSE CV semua kandidat
kandidat = {}
kandidat['Linear Regression'] = df_cv.loc[df_cv['Model'] == 'Linear Regression', 'RMSE CV'].values[0]
for name in tuned:
    kandidat[name] = -tuned[name].best_score_

df_kandidat = pd.Series(kandidat).sort_values().to_frame('RMSE CV')
display(df_kandidat.round(2))

best_name = df_kandidat.index[0]
print('Model terpilih:', best_name)

In [ ]:
# latih model terpilih di seluruh data train
if best_name == 'Linear Regression':
    best_model = models['Linear Regression']
    best_model.fit(X_train, y_train)
else:
    best_model = tuned[best_name].best_estimator_

baseline = models['Dummy (baseline)']
baseline.fit(X_train, y_train)

## 11. Evaluasi di data test

Data test dipakai sekali saja, untuk model terpilih dan baseline.

In [ ]:
def hitung_metrik(model, X_, y_):
    pred = model.predict(X_)
    return {
        'R2': r2_score(y_, pred),
        'MAE ($)': mean_absolute_error(y_, pred),
        'RMSE ($)': np.sqrt(mean_squared_error(y_, pred)),
        'MAPE (%)': np.mean(np.abs((y_ - pred) / y_)) * 100
    }

hasil = pd.DataFrame({
    best_name + ' (train)': hitung_metrik(best_model, X_train, y_train),
    best_name + ' (test)': hitung_metrik(best_model, X_test, y_test),
    'Baseline (test)': hitung_metrik(baseline, X_test, y_test)
}).T
hasil.round(4)

In [ ]:
y_pred = best_model.predict(X_test)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='royalblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Prediksi sempurna')
plt.xlabel('Harga Asli')
plt.ylabel('Harga Prediksi')
plt.title('Harga Asli vs Harga Prediksi - ' + best_name)
plt.legend()
plt.show()

### Analisis residual

Residual = harga asli - harga prediksi. Model yang bagus residualnya tersebar acak di sekitar 0, tidak membentuk pola, dan distribusinya mendekati normal.

In [ ]:
residual = y_test - y_pred

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(y_pred, residual, alpha=0.4)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Harga Prediksi')
plt.ylabel('Residual')
plt.title('Residual vs Prediksi')

plt.subplot(1, 2, 2)
sns.histplot(residual, kde=True)
plt.title('Distribusi Residual')

plt.tight_layout()
plt.show()

In [ ]:
# error per kelompok umur mobil dan per merek (cari bagian yang prediksinya jelek)
cek = X_test.copy()
cek['error'] = np.abs(residual)
cek['kelompok_umur'] = pd.cut(cek['car_age'], bins=[0, 5, 10, 15, 20, 100])

print(cek.groupby('kelompok_umur', observed=True)['error'].mean().round(2))
print()
print(cek.groupby('brand')['error'].mean().round(2))

## 12. Fitur apa yang paling berpengaruh

Pakai permutation importance: satu fitur diacak, lalu dilihat seberapa besar error naik. Cara ini bisa dipakai untuk model apa saja.

In [ ]:
pi = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42,
                            scoring='neg_mean_absolute_error')

importance = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
importance.plot(kind='barh', color='#2b5c8f')
plt.gca().invert_yaxis()
plt.title('Fitur Paling Berpengaruh (kenaikan MAE saat fitur diacak)')
plt.xlabel('Kenaikan MAE ($)')
plt.show()

## 13. Contoh prediksi

In [ ]:
sample_data = X_test.iloc[[0]]
actual_price = y_test.iloc[0]
predicted_price = best_model.predict(sample_data)[0]

print(f"Harga Asli     : ${actual_price:,.2f}")
print(f"Harga Prediksi : ${predicted_price:,.2f}")
print(f"Selisih        : ${abs(actual_price - predicted_price):,.2f}")

## 14. Kesimpulan

Tulis kesimpulan sendiri setelah melihat hasilnya, kira-kira isinya:
- Model terpilih apa dan kenapa (lihat tabel cross validation).
- Hasil di data test (R2, MAE, RMSE) dan seberapa jauh lebih baik dari baseline.
- Apakah tuning membantu mengurangi overfitting.
- 3 fitur paling berpengaruh.

**Keterbatasan:**
- Model hanya cocok untuk rentang data latih (tahun, kapasitas mesin, merek yang ada di data).
- Tidak ada fitur tipe/model mobil, lokasi, kondisi fisik, dan odometer.
- Data dibagi acak, belum memperhitungkan perubahan harga dari waktu ke waktu.
- Cek kembali bagian `Electric` vs `engine_cc` di cleaning.

## 15. Simpan model dan deploy

Yang disimpan: model (pipeline lengkap) dan daftar pilihan input untuk form aplikasi. Aplikasinya pakai Streamlit.

In [ ]:
import os
os.makedirs('deploy', exist_ok=True)

# simpan model
joblib.dump(best_model, 'deploy/model_mobil.pkl')

# simpan info untuk form aplikasi
info = {
    'model': best_name,
    'mae_test': float(hasil.loc[best_name + ' (test)', 'MAE ($)']),
    'r2_test': float(hasil.loc[best_name + ' (test)', 'R2']),
    'pilihan': {col: sorted(df[col].unique().tolist()) for col in kolom_kat},
    'min_max': {col: [float(df[col].min()), float(df[col].max())] for col in kolom_angka},
    'urutan_kolom': X_train.columns.tolist()
}
with open('deploy/info.json', 'w') as f:
    json.dump(info, f, indent=2)

os.listdir('deploy')

In [ ]:
%%writefile deploy/app.py
import streamlit as st
import pandas as pd
import joblib
import json

model = joblib.load('model_mobil.pkl')
with open('info.json') as f:
    info = json.load(f)

pilihan = info['pilihan']
mm = info['min_max']

st.title('Prediksi Harga Mobil Bekas')
st.write('Model:', info['model'], '| R2 test:', round(info['r2_test'], 3), '| MAE test: $', round(info['mae_test']))

brand = st.selectbox('Merek', pilihan['brand'])
fuel_type = st.selectbox('Bahan bakar', pilihan['fuel_type'])
transmission = st.selectbox('Transmisi', pilihan['transmission'])
color = st.selectbox('Warna', pilihan['color'])
service_history = st.selectbox('Riwayat servis', pilihan['service_history'])
insurance_valid = st.selectbox('Asuransi masih berlaku', pilihan['insurance_valid'])

make_year = st.number_input('Tahun pembuatan', 1990, 2026, 2015)
engine_cc = st.number_input('Kapasitas mesin (cc)', int(mm['engine_cc'][0]), int(mm['engine_cc'][1]), 1500)
mileage_kmpl = st.number_input('Efisiensi bahan bakar (km/l)', float(mm['mileage_kmpl'][0]), float(mm['mileage_kmpl'][1]), 18.0)
owner_count = st.number_input('Jumlah pemilik sebelumnya', 1, 10, 2)
accidents_reported = st.number_input('Jumlah kecelakaan', 0, 10, 0)

if st.button('Prediksi'):
    data = pd.DataFrame([{
        'mileage_kmpl': mileage_kmpl,
        'engine_cc': engine_cc,
        'fuel_type': fuel_type,
        'owner_count': owner_count,
        'brand': brand,
        'transmission': transmission,
        'color': color,
        'service_history': service_history,
        'accidents_reported': accidents_reported,
        'insurance_valid': insurance_valid,
        'car_age': 2026 - make_year
    }])
    data = data[info['urutan_kolom']]
    harga = model.predict(data)[0]
    st.success(f'Perkiraan harga: ${harga:,.0f}')
    st.write(f"Kira-kira antara ${harga - info['mae_test']:,.0f} dan ${harga + info['mae_test']:,.0f}")

In [ ]:
# tes: model yang disimpan harus menghasilkan prediksi yang sama
model_load = joblib.load('deploy/model_mobil.pkl')

print(best_model.predict(X_test.head(3)))
print(model_load.predict(X_test.head(3)))

Cara menjalankan aplikasi (di terminal):

```
pip install streamlit
cd deploy
streamlit run app.py
```

Untuk online, upload folder `deploy` ke GitHub lalu deploy lewat Streamlit Community Cloud. Tambahkan file `requirements.txt` yang isinya `scikit-learn`, `pandas`, `joblib`, `streamlit`.